# Django Views

Actividad: Añadir nuevos atributos a ProductModel y actualizar las views del proyecto ecommerce.

Atributos añadidos:
- description
- seller
- color
- product_dimensions


In [ ]:
# models.py
from django.db import models

class ProductModel(models.Model):
    title = models.TextField()
    price = models.FloatField()
    description = models.TextField(blank=True, null=True)
    seller = models.TextField(blank=True, null=True)
    color = models.TextField(blank=True, null=True)
    product_dimensions = models.TextField(blank=True, null=True)


In [ ]:
# forms.py
from django import forms
from .models import ProductModel

class ProductModelForm(forms.ModelForm):
    class Meta:
        model = ProductModel
        fields = [
            "title",
            "price",
            "description",
            "seller",
            "color",
            "product_dimensions"
        ]


In [ ]:
# views.py
from django.contrib.auth.decorators import login_required
from django.contrib import messages
from django.db.models import Q
from django.shortcuts import render, get_object_or_404
from django.http import HttpResponseRedirect

from .forms import ProductModelForm
from .models import ProductModel

def product_model_delete_view(request, product_id):
    instance = get_object_or_404(ProductModel, id=product_id)
    if request.method == "POST":
        instance.delete()
        messages.success(request, "Producto eliminado")
        return HttpResponseRedirect("/ecommerce/")
    context = {"product": instance}
    template = "ecommerce/delete-view.html"
    return render(request, template, context)

def product_model_update_view(request, product_id=None):
    instance = get_object_or_404(ProductModel, id=product_id)
    form = ProductModelForm(request.POST or None, instance=instance)
    if form.is_valid():
        instance = form.save(commit=False)
        instance.save()
        messages.success(request, "Producto actualizado con éxito")
        return HttpResponseRedirect(f"/ecommerce/{instance.id}")
    context = {"form": form}
    template = "ecommerce/update-view.html"
    return render(request, template, context)

def product_model_create_view(request):
    form = ProductModelForm(request.POST or None)
    if form.is_valid():
        instance = form.save(commit=False)
        instance.save()
        messages.success(request, "Producto creado con éxito")
        return HttpResponseRedirect(f"/ecommerce/{instance.id}")
    context = {"form": form}
    template = "ecommerce/create-view.html"
    return render(request, template, context)

def product_model_detail_view(request, product_id):
    instance = get_object_or_404(ProductModel, id=product_id)
    context = {"product": instance}
    template = "ecommerce/detail-view.html"
    return render(request, template, context)

def product_model_list_view(request):
    query = request.GET.get("q", None)
    queryset = ProductModel.objects.all()
    if query is not None:
        queryset = queryset.filter(
            Q(title__icontains=query) | Q(price__icontains=query)
        )
    context = {"products": queryset}
    template = "ecommerce/list-view.html"
    return render(request, template, context)


In [ ]:
# create-view.html
{% include "ecommerce/messages.html" %}
<h1>Crear un nuevo producto</h1>
<form method="POST" action="">
    {% csrf_token %}
    {{form.as_p}}
    <input type="submit" value="Crear">
</form>


In [ ]:
# update-view.html
<h1>Actualización de producto {{ form.instance.title}}</h1>
{{ form.instance.title }}
<form method="POST" action="">
    {% csrf_token %}
    {{ form.as_p }}
    <input type="submit" value="Actualizar">
</form>


In [ ]:
# delete-view.html
{% include "ecommerce/messages.html" %}
<h1>Eliminar - {{ product.title }}</h1>
<p>Precio: {{ product.price }}</p>
<p>Color: {{ product.color }}</p>
<p>Vendedor: {{ product.seller }}</p>
<form method="POST" action="">
    {% csrf_token %}
    ¿Estás seguro que quieres eliminar el producto?
    <input type="submit" value="Eliminar">
    <a href="/ecommerce/{{ product.id }}">Cancelar</a>
</form>


In [ ]:
# detail-view.html
<h1>{{ product.title }}</h1>
<p>Precio: {{ product.price }}</p>
<p>Descripción: {{ product.description }}</p>
<p>Vendedor: {{ product.seller }}</p>
<p>Color: {{ product.color }}</p>
<p>Dimensiones: {{ product.product_dimensions }}</p>


In [ ]:
# list-view.html
{% include "ecommerce/search.html" %}
{% include "ecommerce/messages.html" %}
<h1>Vista de listado</h1>
{% for product in products %}
    <li><a href="/ecommerce/{{ product.id }}">{{ product.title }}</a> - {{ product.price }} - {{ product.color }}</li>
{% endfor %}
